In [10]:
import pandas as pd
from ml_enhance import QuantumFPFileLoader, parallelize
import gzip
import json
from tqdm import tqdm
from rdkit import Chem

In [3]:
df = pd.read_csv("data/processed_dataset.csv")

In [4]:
smiles, canon_smiles = df["smiles"], df["canon_smiles"]
lookup_table = pd.DataFrame({"smiles": smiles, "canon_smiles": canon_smiles})

In [5]:
loader = QuantumFPFileLoader("data/QuantumFP/QFP_output")
output_files = loader.list_output_files()

In [ ]:
def get_smiles_ids(file):
    

In [9]:
calc_ids = []
output_smiles = []
for file in tqdm(output_files):
    with gzip.open(file, "rt") as f:
        data: list[dict] = json.load(f)

        output_smiles.append(data[0]["original_smiles"])
        calc_ids.append(data[0]["id"])

  0%|          | 22/8837 [00:14<1:35:04,  1.55it/s]


KeyboardInterrupt: 

In [ ]:
lookup_table["calc_id"] = calc_ids

In [ ]:
def canonicalize_smiles(smiles: str) -> str | None:
    if not isinstance(smiles, str):
        return None

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    for atom in mol.GetAtoms():
        atom.SetAtomMapNum(0)

    return Chem.MolToSmiles(mol, canonical=True)

In [ ]:
raw_data = pd.read_csv("data/AqSolDB/data_curated.csv")
test_df = raw_data[["SMILES", "ID"]].copy()
raw_smiles = raw_data["SMILES"]
db_ids = raw_data["ID"]

test_df["canon_smiles"] = test_df["SMILES"].apply(canonicalize_smiles)
test_df = test_df.dropna(subset=["canon_smiles"])

id_filter = test_df.set_index("canon_smiles")["ID"]

lookup_table["db_id"] = lookup_table["canon_smiles"].map(id_filter)

In [ ]:
lookup_table